In [ ]:
# ============================================================
#  LLM POWERED CODE REVIEW ASSISTANT
#  Generative AI (NLP) | LLM Fine-tuning Concepts
#  Single cell — paste and run in JupyterLab (Shift+Enter)
# ============================================================

# ---------- 0. Install dependencies ----------
import subprocess, sys
for pkg in ["anthropic", "ipywidgets", "IPython"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import anthropic, os, textwrap
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ---------- 1. API Key ----------
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not API_KEY:
    API_KEY = input("Enter your Anthropic API key: ").strip()
client = anthropic.Anthropic(api_key=API_KEY)

# ---------- 2. System Prompt (acts like fine-tuning instructions) ----------
def build_system_prompt(language, focus):
    return textwrap.dedent(f"""\
    You are a senior software engineer specializing in {language} code reviews.
    Your review focus: {focus}.

    You review code like a principal engineer at a top tech company.
    You are thorough, constructive, and educational.

    Always respond in this EXACT structure:

    SCORE: [0-100]
    GRADE: [A/B/C/D]
    ISSUES: [count]
    SUGGESTIONS: [count]

    SUMMARY
    -------
    [2-3 sentence overall assessment]

    ISSUES FOUND
    ------------
    [Each issue with: severity (CRITICAL/WARNING/INFO), location, explanation]

    SUGGESTIONS
    -----------
    [Numbered, concrete, actionable improvements]

    IMPROVED CODE
    -------------
    [Rewritten version with inline comments explaining each fix]

    LEARNING NOTES
    --------------
    [1-2 key concepts the developer should study based on the issues found]
    """)

# ---------- 3. Core review function ----------
def review_code(code: str, language: str = "Python",
                focus: str = "Full review (all aspects)",
                stream: bool = True) -> str:
    """
    Send code to Claude for review. Returns the full review as a string.
    """
    system = build_system_prompt(language, focus)
    user_msg = f"Please review this {language} code:\n\n```{language.lower()}\n{code}\n```"

    if stream:
        parts = []
        with client.messages.stream(
            model="claude-sonnet-4-20250514",
            max_tokens=2000,
            system=system,
            messages=[{"role": "user", "content": user_msg}]
        ) as s:
            for chunk in s.text_stream:
                print(chunk, end="", flush=True)
                parts.append(chunk)
        print()
        return "".join(parts)
    else:
        msg = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=2000,
            system=system,
            messages=[{"role": "user", "content": user_msg}]
        )
        return msg.content[0].text

# ---------- 4. Example code snippets ----------
EXAMPLES = {
    "Python — Basic function": ("Python", """\
def calculate_total(items):
    total = 0
    for i in range(len(items)):
        total = total + items[i]['price']
    return total

def get_user(db, user_id):
    query = "SELECT * FROM users WHERE id = " + user_id
    return db.execute(query)
"""),
    "Python — Security issues": ("Python", """\
import subprocess

def run_command(user_input):
    cmd = "ls " + user_input
    result = subprocess.run(cmd, shell=True, capture_output=True)
    return result.stdout

def login(username, password):
    if username == "admin" and password == "admin123":
        return True
    return False

def read_file(filename):
    with open("/var/data/" + filename) as f:
        return f.read()
"""),
    "JavaScript — Async code": ("JavaScript", """\
async function fetchData(url) {
  const response = await fetch(url)
  const data = response.json()
  console.log(data)
  return data
}

function processUsers(users) {
  var result = []
  for(var i=0; i<users.length; i++) {
    result.push(users[i].name.toUpperCase())
  }
  return result
}
"""),
    "SQL — Query optimization": ("SQL", """\
SELECT u.name, COUNT(*) 
FROM users u, orders o
WHERE u.id = o.user_id 
AND o.status = 'pending'
GROUP BY u.name
ORDER BY COUNT(*) DESC
"""),
}

# ---------- 5. Widget UI ----------
style = {"description_width": "initial"}

title_html = widgets.HTML(
    value="<h2 style='margin:0 0 4px;font-size:20px;'>LLM Code Review Assistant</h2>"
           "<p style='color:#666;font-size:13px;margin:0 0 12px;'>Powered by Claude · Generative AI (NLP)</p>"
)

lang_select = widgets.Dropdown(
    options=["Python","JavaScript","Java","C++","SQL","TypeScript","Go","Rust","Other"],
    value="Python", description="Language:",
    layout=widgets.Layout(width="48%"), style=style
)

focus_select = widgets.Dropdown(
    options=[
        "Full review (all aspects)",
        "Security vulnerabilities",
        "Performance & optimization",
        "Code style & readability",
        "Bug detection",
        "Best practices & patterns",
    ],
    value="Full review (all aspects)",
    description="Focus:",
    layout=widgets.Layout(width="48%"), style=style
)

example_select = widgets.Dropdown(
    options=[""] + list(EXAMPLES.keys()),
    value="", description="Load example:",
    layout=widgets.Layout(width="100%"), style=style
)

code_input = widgets.Textarea(
    placeholder="Paste your code here...",
    layout=widgets.Layout(width="100%", height="220px"),
    style=style
)

stream_toggle = widgets.Checkbox(value=True, description="Stream output", style=style)
review_btn = widgets.Button(description="Review Code", button_style="primary",
                             layout=widgets.Layout(width="140px"))
clear_btn  = widgets.Button(description="Clear", button_style="",
                             layout=widgets.Layout(width="80px"))

score_html = widgets.HTML(value="")
output_area = widgets.Output(layout=widgets.Layout(
    border="1px solid #ddd", padding="12px",
    border_radius="8px", min_height="80px", width="100%"
))

last_review = {"value": ""}

def on_example(change):
    if change["new"] and change["new"] in EXAMPLES:
        lang, code = EXAMPLES[change["new"]]
        code_input.value = code
        lang_select.value = lang
        example_select.value = ""

def parse_metrics(text):
    import re
    score = re.search(r'SCORE:\s*(\d+)', text)
    grade = re.search(r'GRADE:\s*([ABCD])', text)
    issues = re.search(r'ISSUES:\s*(\d+)', text)
    sugg = re.search(r'SUGGESTIONS:\s*(\d+)', text)
    colors = {'A':'#1D9E75','B':'#378ADD','C':'#EF9F27','D':'#E24B4A'}
    if score:
        g = grade.group(1) if grade else 'B'
        c = colors.get(g, '#378ADD')
        return (
            f"<div style='display:flex;gap:12px;flex-wrap:wrap;margin:8px 0;'>"
            f"<div style='background:#f6f8fa;border-radius:8px;padding:10px 16px;text-align:center;min-width:80px'>"
            f"<div style='font-size:22px;font-weight:600;color:{c}'>{score.group(1)}/100</div>"
            f"<div style='font-size:11px;color:#888'>Quality score</div></div>"
            f"<div style='background:#f6f8fa;border-radius:8px;padding:10px 16px;text-align:center;min-width:80px'>"
            f"<div style='font-size:22px;font-weight:600;color:{c}'>{g}</div>"
            f"<div style='font-size:11px;color:#888'>Grade</div></div>"
            f"<div style='background:#f6f8fa;border-radius:8px;padding:10px 16px;text-align:center;min-width:80px'>"
            f"<div style='font-size:22px;font-weight:600'>{issues.group(1) if issues else '—'}</div>"
            f"<div style='font-size:11px;color:#888'>Issues</div></div>"
            f"<div style='background:#f6f8fa;border-radius:8px;padding:10px 16px;text-align:center;min-width:80px'>"
            f"<div style='font-size:22px;font-weight:600'>{sugg.group(1) if sugg else '—'}</div>"
            f"<div style='font-size:11px;color:#888'>Suggestions</div></div>"
            f"</div>"
        )
    return ""

def on_review(b):
    code = code_input.value.strip()
    if not code:
        with output_area:
            clear_output()
            print("Please paste some code first.")
        return
    review_btn.disabled = True
    review_btn.description = "Reviewing..."
    score_html.value = ""
    with output_area:
        clear_output()
        sep = "─" * 60
        print(f"Language: {lang_select.value}  |  Focus: {focus_select.value}")
        print(sep)
        result = review_code(
            code,
            language=lang_select.value,
            focus=focus_select.value,
            stream=stream_toggle.value
        )
        last_review["value"] = result
        score_html.value = parse_metrics(result)
    review_btn.disabled = False
    review_btn.description = "Review Code"

def on_clear(b):
    code_input.value = ""
    score_html.value = ""
    with output_area:
        clear_output()
    last_review["value"] = ""

example_select.observe(on_example, names="value")
review_btn.on_click(on_review)
clear_btn.on_click(on_clear)

selects_row = widgets.HBox([lang_select, focus_select],
                            layout=widgets.Layout(gap="12px", width="100%"))
btn_row = widgets.HBox([review_btn, clear_btn, stream_toggle],
                        layout=widgets.Layout(gap="8px", align_items="center"))

ui = widgets.VBox([
    title_html,
    selects_row,
    example_select,
    widgets.HTML(value="<b style='font-size:13px'>Your code</b>"),
    code_input,
    btn_row,
    score_html,
    output_area,
], layout=widgets.Layout(width="100%", gap="8px"))

display(ui)

# ---------- 6. Programmatic usage (no UI) ----------
# Uncomment to use directly:
# result = review_code("""
# def add(a, b):
#     return a+b
# """, language='Python', focus='Best practices')
# print(result)
